<a href="https://colab.research.google.com/github/Rds1007/SQL_BigDataInterview/blob/main/Salting_skew.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,rand,concat_ws
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,TimestampType,DecimalType
spark=SparkSession.builder.appName("saltingexample").getOrCreate()

In [ ]:
##reading transation data
tr_schema=StructType([StructField("transaction_id", IntegerType(),True),StructField("customer_id", IntegerType(),True) ,StructField("amount", DecimalType(10, 2),True), # Changed to DecimalType(10, 2) for 2 decimal places
    StructField("transaction_time", TimestampType(),True)]    )
cust_schema=StructType([StructField("custmer_id",IntegerType(),True),StructField("name",StringType(),True)])


In [ ]:

trasactions_df=spark.read.csv("/content/sample_data/transactions.csv",header=True, schema=tr_schema)

In [ ]:
customer_df=spark.read.csv("/content/sample_data/customers.csv",header=True,schema=cust_schema)

In [ ]:
from pyspark.sql.functions import concat_ws, col, rand,floor
salt_count=10
#adding salt to the transactions_df
trasactions_df=trasactions_df.withColumn("salt",floor(rand()*salt_count))
trasactions_df=trasactions_df.withColumn("salted_key",concat_ws("_","customer_id","salt")).drop("salt")
#adding salt to the customer_df


In [ ]:
from pyspark.sql.functions import array, lit,col, rand,explode
customer_df=customer_df.withColumn("salt",explode(array(*[lit(i) for i in range(salt_count)])))
customer_df=customer_df.withColumn("salted_key",concat_ws("_","custmer_id","salt")).drop("salt")

In [ ]:
result_joined=trasactions_df.join(customer_df,trasactions_df.salted_key==customer_df.salted_key,"left")

In [ ]:
result_joined.filter(col("name").isNotNull()).show()

+--------------+-----------+------+-------------------+----------+----------+-------------+----------+
|transaction_id|customer_id|amount|   transaction_time|salted_key|custmer_id|         name|salted_key|
+--------------+-----------+------+-------------------+----------+----------+-------------+----------+
|             1|        410|744.13|2026-01-01 00:00:01|     410_2|       410| Customer_410|     410_2|
|             2|        100|148.14|2026-01-01 00:00:02|     100_8|       100| Customer_100|     100_8|
|             3|        100|743.26|2026-01-01 00:00:03|     100_5|       100| Customer_100|     100_5|
|             4|       9675|427.70|2026-01-01 00:00:04|    9675_2|      9675|Customer_9675|    9675_2|
|             5|        100|226.45|2026-01-01 00:00:05|     100_0|       100| Customer_100|     100_0|
|             6|        435|565.63|2026-01-01 00:00:06|     435_8|       435| Customer_435|     435_8|
|             7|       8929|425.32|2026-01-01 00:00:07|    8929_8|      8